# T5 Encode → Pool → Decode Roundtrip

Take one short and one detailed caption from stimulus 0, encode with `google/t5-v1_1-xxl`
(the same model FLUX.1-dev uses as `text_encoder_2`, hidden dim 4096), pool to a single
vector, then try to recover the original text by decoding the pooled vector.
Measure cosine similarity between original and recovered embeddings.

In [2]:
import torch
import torch.nn.functional as F
from transformers import T5Tokenizer, T5ForConditionalGeneration
from transformers.modeling_outputs import BaseModelOutput

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


In [3]:
# The two captions for stimulus 0
short_caption    = "two mailboxes in front of a house"
detailed_caption = (
    "a pair of mailboxes on a sidewalk in front of a house. "
    "The setting is outdoors, near a house. "
    "The notable visual details include the black and white color scheme, "
    "the presence of mailboxes, and their location on the sidewalk in front of a house"
)

print("Short   :", short_caption)
print("Detailed:", detailed_caption)

Short   : two mailboxes in front of a house
Detailed: a pair of mailboxes on a sidewalk in front of a house. The setting is outdoors, near a house. The notable visual details include the black and white color scheme, the presence of mailboxes, and their location on the sidewalk in front of a house


In [4]:
# Load google/t5-v1_1-xxl with bfloat16 + device_map to avoid OOM
# xxl is ~11GB; bfloat16 halves that. device_map skips the CPU->GPU copy spike.
MODEL_NAME = "google/t5-v1_1-xxl"

tokenizer  = T5Tokenizer.from_pretrained(MODEL_NAME)
full_model = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
).to(device).eval()
# Reuse the encoder from full_model — no second copy needed
encoder = full_model.encoder

print("Loaded", MODEL_NAME)
print("Hidden dim:", full_model.config.d_model)
print("dtype:", next(full_model.parameters()).dtype)

Loading weights:   0%|          | 0/560 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loaded google/t5-v1_1-xxl
Hidden dim: 4096
dtype: torch.bfloat16


In [22]:
def encode_caption(text: str) -> dict:
    """Tokenize → encoder → return token embeddings and two pooled vectors."""
    tokens = tokenizer(
        text,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=512,
    )
    # Move tokens to the device where the first encoder layer lives
    first_device = next(encoder.parameters()).device
    tokens = {k: v.to(first_device) for k, v in tokens.items()}

    with torch.no_grad():
        out = encoder(**tokens)

    hidden = out.last_hidden_state.float()  # (1, seq_len, 4096)
    mask   = tokens['attention_mask'].unsqueeze(-1).float()  # (1, seq_len, 1)

    # Mean pooling (masked)
    mean_pooled = (hidden * mask).sum(1) / mask.sum(1)  # (1, 4096)

    # Norm-weighted softmax pooling (same as project data_utils)
    norms   = hidden.norm(dim=-1)                          # (1, seq_len)
    weights = torch.softmax(norms, dim=-1).unsqueeze(-1)   # (1, seq_len, 1)
    norm_pooled = (weights * hidden).sum(1)                # (1, 4096)

    return {
        'tokens':      tokens,
        'hidden':      hidden,          # (1, seq, 4096)
        'mean_pooled': mean_pooled,     # (1, 4096)
        'norm_pooled': norm_pooled,     # (1, 4096)
        'n_tokens':    int(mask.sum().item()),
    }


enc_short    = encode_caption(short_caption)
enc_detailed = encode_caption(detailed_caption)

print(f"Short    → {enc_short['n_tokens']} tokens, hidden {enc_short['hidden'].shape}")
print(f"Detailed → {enc_detailed['n_tokens']} tokens, hidden {enc_detailed['hidden'].shape}")

Short    → 11 tokens, hidden torch.Size([1, 11, 4096])
Detailed → 60 tokens, hidden torch.Size([1, 60, 4096])


In [23]:
# Verify: compare our encoder output against FLUX-cached t5_embeds.pt for stimulus 0
# cache/t5_embeds.pt was produced by pipe.encode_prompt() using the same xxl model.
# If they match, we're in the same embedding space.

import sys, torch, torch.nn.functional as F
sys.path.insert(0, '/home/yy3658/NeurObjectGen')

T5_CACHE = '/home/yy3658/NeurObjectGen/cache/t5_embeds.pt'
flux_t5 = torch.load(T5_CACHE, weights_only=True).float()  # (300, 512, 4096)
flux_t5_0 = flux_t5[0]  # (512, 4096) — stimulus 0, short caption
print('Loaded FLUX cache:', flux_t5.shape)

# Encode stimulus 0 short caption with our model
enc = encode_caption(short_caption)
our_hidden = enc['hidden'][0]  # (seq, 4096), float32
n = enc['n_tokens']
print(f'Our encoder: {our_hidden.shape}, {n} real tokens')

# FLUX pads to 512; compare token-by-token on the real (non-pad) tokens
flux_tokens = flux_t5_0[:n].to(our_hidden.device)  # (n, 4096)
our_tokens  = our_hidden[:n]                        # (n, 4096)

cos_per_tok = F.cosine_similarity(our_tokens, flux_tokens, dim=-1)  # (n,)
print(f'Per-token cosine similarity (mean): {cos_per_tok.mean().item():.6f}')
print(f'Per-token cosine similarity (min):  {cos_per_tok.min().item():.6f}')
print(f'L2 diff (mean per token):           {(our_tokens - flux_tokens).norm(dim=-1).mean().item():.6f}')
print()
if cos_per_tok.mean().item() > 0.9999:
    print('MATCH — same embedding space confirmed')
else:
    print('MISMATCH — embeddings differ, check tokenization or model weights')

Loaded FLUX cache: torch.Size([300, 512, 4096])
Our encoder: torch.Size([11, 4096]), 11 real tokens
Per-token cosine similarity (mean): 0.547999
Per-token cosine similarity (min):  -0.150509
L2 diff (mean per token):           5.512971

MISMATCH — embeddings differ, check tokenization or model weights


In [24]:
def decode_from_pooled(pooled_vec: torch.Tensor, max_new_tokens: int = 60) -> str:
    """pooled_vec: (1, 4096) float32"""
    dec_device = next(full_model.decoder.parameters()).device
    enc_hidden = pooled_vec.to(dec_device).to(torch.bfloat16).unsqueeze(1)  # (1, 1, 4096)
    enc_mask   = torch.ones(1, 1, dtype=torch.long, device=dec_device)

    with torch.no_grad():
        out = full_model.generate(
            encoder_outputs=BaseModelOutput(last_hidden_state=enc_hidden),
            attention_mask=enc_mask,
            max_new_tokens=60,
            num_beams=4,
        )

    return tokenizer.decode(out[0], skip_special_tokens=True)


print("Decoding short caption (mean pooled)...")
recovered_short_mean    = decode_from_pooled(enc_short['mean_pooled'])
print("Decoding short caption (norm pooled)...")
recovered_short_norm    = decode_from_pooled(enc_short['norm_pooled'])
print("Decoding detailed caption (mean pooled)...")
recovered_detailed_mean = decode_from_pooled(enc_detailed['mean_pooled'])
print("Decoding detailed caption (norm pooled)...")
recovered_detailed_norm = decode_from_pooled(enc_detailed['norm_pooled'])

print()
print('=== Recovered texts ===')
print(f'Short    (mean): {recovered_short_mean}')
print(f'Short    (norm): {recovered_short_norm}')
print(f'Detailed (mean): {recovered_detailed_mean}')
print(f'Detailed (norm): {recovered_detailed_norm}')

Decoding short caption (mean pooled)...
Decoding short caption (norm pooled)...
Decoding detailed caption (mean pooled)...
Decoding detailed caption (norm pooled)...

=== Recovered texts ===
Short    (mean): mailboxes in a row of four mailboxes in a row of four mailboxes in a row of four mailboxes in a mailbox with a mailbox mailbox a mailbox a mailbox a mailbox a mailbox a
Short    (norm): and a mailbox with a mailbox and a mailbox with a mailbox with a mailbox and a mailbox with a mailbox with a mailbox with a mailbox with a mailbox with a mailbox with a mailbox with a mailbox with a mailbox with a
Detailed (mean): ne, a type ofne, with a building in the foreground and a sign in the foreground. The setting is a sidewalk in front of a building and in front of a building, in on a street with a
Detailed (norm): in front of a building. The front of the building is in front of a building. The setting is in front of a building. The setting is in front of a building in front of a building i

In [25]:
def cosine_sim(a: torch.Tensor, b: torch.Tensor) -> float:
    return F.cosine_similarity(a, b).item()


def embedding_similarity(original_enc: dict, recovered_text: str, label: str):
    """Encode the recovered text and compare its pooled vector to the original."""
    rec_enc = encode_caption(recovered_text)
    sim_mean = cosine_sim(original_enc["mean_pooled"], rec_enc["mean_pooled"])
    sim_norm = cosine_sim(original_enc["norm_pooled"], rec_enc["norm_pooled"])
    print(f"{label}")
    print(f"  cosine (mean pool): {sim_mean:.4f}")
    print(f"  cosine (norm pool): {sim_norm:.4f}")
    return sim_mean, sim_norm


print("=== Embedding similarity: original vs recovered ===")
print()
embedding_similarity(enc_short,    recovered_short_mean,    "Short caption    | decoded from mean pool")
print()
embedding_similarity(enc_short,    recovered_short_norm,    "Short caption    | decoded from norm pool")
print()
embedding_similarity(enc_detailed, recovered_detailed_mean, "Detailed caption | decoded from mean pool")
print()
embedding_similarity(enc_detailed, recovered_detailed_norm, "Detailed caption | decoded from norm pool")

=== Embedding similarity: original vs recovered ===

Short caption    | decoded from mean pool
  cosine (mean pool): 0.4993
  cosine (norm pool): 0.4468

Short caption    | decoded from norm pool
  cosine (mean pool): 0.4132
  cosine (norm pool): 0.3815

Detailed caption | decoded from mean pool
  cosine (mean pool): 0.8632
  cosine (norm pool): 0.8469

Detailed caption | decoded from norm pool
  cosine (mean pool): 0.7819
  cosine (norm pool): 0.7529


(0.7819404602050781, 0.7528654932975769)

In [26]:
# Baseline: cross-caption similarity (short vs detailed) for reference
print("=== Baseline cross-caption similarities ===")
print(f"Short mean_pool vs Detailed mean_pool: {cosine_sim(enc_short['mean_pooled'], enc_detailed['mean_pooled']):.4f}")
print(f"Short norm_pool vs Detailed norm_pool: {cosine_sim(enc_short['norm_pooled'], enc_detailed['norm_pooled']):.4f}")

=== Baseline cross-caption similarities ===
Short mean_pool vs Detailed mean_pool: 0.6185
Short norm_pool vs Detailed norm_pool: 0.6185


In [27]:
# Broadcasting loss: pooled → broadcast (1, 512, 4096) vs FLUX full sequence
# We use the FLUX cache as the reference 'true' full-sequence embedding.
# Note: our encoder has a numeric offset vs FLUX cache (different padding),
# so we measure the loss of broadcasting *within* each system consistently:
# i.e. pool our own hidden states, broadcast back, compare to our own originals.

import json
import torch
import torch.nn.functional as F

N_STIMULI = 20

with open('/home/yy3658/NeurObjectGen/cache/blip2_captions.json') as f:
    short_caps = json.load(f)
with open('/home/yy3658/NeurObjectGen/cache/blip2_detailed_captions.json') as f:
    detailed_caps = json.load(f)

captions = [(short_caps[str(i)], detailed_caps[str(i)]) for i in range(N_STIMULI)]

def pool_and_broadcast(hidden, mask, seq_len=512):
    """hidden: (1, T, 4096), mask: (1, T, 1) → broadcast pooled to (1, seq_len, 4096)"""
    mean_p = (hidden * mask).sum(1) / mask.sum(1)          # (1, 4096)
    norms   = hidden.norm(dim=-1)
    weights = torch.softmax(norms, dim=-1).unsqueeze(-1)
    norm_p  = (weights * hidden).sum(1)                    # (1, 4096)
    mean_bc = mean_p.unsqueeze(1).expand(-1, seq_len, -1)  # (1, 512, 4096)
    norm_bc = norm_p.unsqueeze(1).expand(-1, seq_len, -1)
    return mean_bc, norm_bc

def seq_cosine(a, b, n_tokens):
    """Mean cosine similarity over real (non-pad) tokens."""
    return F.cosine_similarity(a[0, :n_tokens], b[0, :n_tokens], dim=-1).mean().item()

results = []
for i, (short, detailed) in enumerate(captions):
    for cap_type, caption in [('short', short), ('detailed', detailed)]:
        enc = encode_caption(caption)
        hidden = enc['hidden']                                   # (1, T, 4096)
        mask   = enc['tokens']['attention_mask'].unsqueeze(-1).float()
        n      = enc['n_tokens']
        mean_bc, norm_bc = pool_and_broadcast(hidden, mask, seq_len=n)
        cos_mean = seq_cosine(mean_bc, hidden, n)
        cos_norm = seq_cosine(norm_bc, hidden, n)
        results.append(dict(i=i, type=cap_type, n_tokens=n,
                            cos_mean=cos_mean, cos_norm=cos_norm))

import statistics
for cap_type in ['short', 'detailed']:
    subset = [r for r in results if r['type'] == cap_type]
    mean_cos = statistics.mean(r['cos_mean'] for r in subset)
    norm_cos = statistics.mean(r['cos_norm'] for r in subset)
    avg_tok  = statistics.mean(r['n_tokens'] for r in subset)
    print(f"{cap_type:8s} | avg_tokens={avg_tok:.1f} | "
          f"broadcast_cos (mean pool)={mean_cos:.4f} | "
          f"broadcast_cos (norm pool)={norm_cos:.4f}")

print()
print('Per-stimulus breakdown:')
print(f'{"idx":>4} {"type":>8} {"tok":>5} {"mean_bc":>9} {"norm_bc":>9}')
for r in results:
    print(f"{r['i']:>4} {r['type']:>8} {r['n_tokens']:>5} "
          f"{r['cos_mean']:>9.4f} {r['cos_norm']:>9.4f}")

short    | avg_tokens=10.2 | broadcast_cos (mean pool)=0.4359 | broadcast_cos (norm pool)=0.3868
detailed | avg_tokens=56.4 | broadcast_cos (mean pool)=0.4703 | broadcast_cos (norm pool)=0.4569

Per-stimulus breakdown:
 idx     type   tok   mean_bc   norm_bc
   0    short    11    0.4084    0.3511
   0 detailed    60    0.4919    0.4854
   1    short     8    0.4062    0.3593
   1 detailed    54    0.4629    0.4415
   2    short    16    0.3939    0.3397
   2 detailed    55    0.4804    0.4602
   3    short     7    0.4442    0.3827
   3 detailed    58    0.4796    0.4682
   4    short    10    0.4404    0.3601
   4 detailed    64    0.4604    0.4450
   5    short     7    0.4291    0.3792
   5 detailed    46    0.4483    0.4361
   6    short     7    0.4546    0.3901
   6 detailed    60    0.4699    0.4601
   7    short    17    0.3770    0.3516
   7 detailed    62    0.4751    0.4651
   8    short     7    0.4343    0.3890
   8 detailed    53    0.4864    0.4649
   9    short     6  

In [28]:
# PCA variance explained on the full FLUX T5 sequence embeddings
# t5_embeds.pt: (300, 512, 4096) — 300 stimuli, 512 tokens, 4096 dims
# Flatten to (300*512, 4096), run PCA, check cumulative variance at K=64 and beyond.

import torch

T5_CACHE = '/home/yy3658/NeurObjectGen/cache/t5_embeds.pt'
t5 = torch.load(T5_CACHE, weights_only=True).float()  # (300, 512, 4096)
print('Loaded:', t5.shape)

# Flatten stimuli×tokens into one big matrix
X = t5.reshape(-1, 4096)  # (153600, 4096)
X = X - X.mean(dim=0)     # center

# SVD-based PCA (randomized for speed)
print('Running randomized SVD (k=128)...')
U, S, Vh = torch.pca_lowrank(X, q=128, niter=4)
var_explained = (S ** 2) / (S ** 2).sum()
cumvar = var_explained.cumsum(0)

print()
print(f'  K   | var_explained | cumulative')
print(f'------+---------------+-----------')
for k in [1, 2, 4, 8, 16, 32, 64, 96, 128]:
    print(f'  {k:>3} | {var_explained[:k].sum().item()*100:>11.2f}% | {cumvar[k-1].item()*100:>8.2f}%')

Loaded: torch.Size([300, 512, 4096])
Running randomized SVD (k=128)...

  K   | var_explained | cumulative
------+---------------+-----------
    1 |       49.56% |    49.56%
    2 |       69.83% |    69.83%
    4 |       80.82% |    80.82%
    8 |       89.22% |    89.22%
   16 |       94.10% |    94.10%
   32 |       96.81% |    96.81%
   64 |       98.64% |    98.64%
   96 |       99.48% |    99.48%
  128 |      100.00% |   100.00%


In [29]:
# PCA recovery: replicate the exact project pipeline and measure roundtrip similarity
# Pipeline: (300, 512, 4096) → mean-pool → (300, 4096) → project K coords → reconstruct → cosine sim
# Uses the cached basis/mean from precompute_t5_pca.py

import torch, torch.nn.functional as F
from pathlib import Path

CACHE = Path('/home/yy3658/NeurObjectGen/cache')
K = 64

t5      = torch.load(CACHE / 't5_embeds.pt', weights_only=True).float()           # (300, 512, 4096)
basis   = torch.load(CACHE / f't5_pca_basis_k{K}.pt', weights_only=True).float() # (K, 4096)
mean    = torch.load(CACHE / f't5_pca_mean_k{K}.pt',  weights_only=True).float() # (4096,)

# Exact project pooling from data_utils/rust_loader.py
norms   = t5.norm(dim=-1)                              # (300, 512)
weights = torch.softmax(norms, dim=-1).unsqueeze(-1)   # (300, 512, 1)
pooled  = (weights * t5).sum(dim=1) - mean             # (300, 4096) centered

# Project → reconstruct
coords  = pooled @ basis.T   # (300, K)
recon   = coords @ basis     # (300, 4096)

# Cosine similarity between original pooled (centered) and reconstructed
cos     = F.cosine_similarity(pooled, recon, dim=-1)   # (300,)
l2      = (pooled - recon).norm(dim=-1)                # (300,)

print(f'=== PCA roundtrip (K={K}, norm-weighted pool, short captions) ===')
print(f'Cosine similarity — mean: {cos.mean():.4f}  min: {cos.min():.4f}  max: {cos.max():.4f}')
print(f'L2 error          — mean: {l2.mean():.4f}  min: {l2.min():.4f}  max: {l2.max():.4f}')
print()

# Variance explained (from singular values of the *centered* pooled matrix)
_, S, _ = torch.linalg.svd(pooled, full_matrices=False)
var_total = S.pow(2).sum()
for k in [1, 8, 16, 32, 64]:
    pct = 100 * S[:k].pow(2).sum() / var_total
    print(f'  K={k:>3}: {pct:.1f}% variance explained')

# Also show worst and best stimuli
print()
worst = cos.argmin().item()
best  = cos.argmax().item()
print(f'Best  stimulus {best:>3}: cosine={cos[best]:.4f}')
print(f'Worst stimulus {worst:>3}: cosine={cos[worst]:.4f}')

=== PCA roundtrip (K=64, norm-weighted pool, short captions) ===
Cosine similarity — mean: 0.8731  min: 0.7426  max: 0.9648
L2 error          — mean: 1.5689  min: 0.5776  max: 3.3176

  K=  1: 77.8% variance explained
  K=  8: 95.2% variance explained
  K= 16: 97.2% variance explained
  K= 32: 98.5% variance explained
  K= 64: 99.3% variance explained

Best  stimulus 296: cosine=0.9648
Worst stimulus 257: cosine=0.7426
